# 01. LangGraph 기초 — StateGraph & Multi-Agent

## 학습 목표
1. LangGraph의 핵심 개념 직접 구현 (StateGraph, Node, Edge, Conditional Edge)
2. ReAct 패턴 에이전트 구현 (Reasoning + Acting)
3. 단일 에이전트 → 멀티 에이전트 확장
4. Human-in-the-loop 패턴

## 핵심 개념
- **State**: 에이전트 간 공유되는 딕셔너리 (그래프 전체의 메모리)
- **Node**: 상태를 받아 처리하고 업데이트된 상태를 반환하는 함수
- **Edge**: 노드 간 연결 (조건부 분기 포함)
- **Checkpointer**: 상태 스냅샷 저장 → Human-in-the-loop 가능

In [1]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os
from dotenv import load_dotenv
load_dotenv('../.env')

print('ANTHROPIC_API_KEY 로드:', 'OK' if os.getenv('ANTHROPIC_API_KEY') else 'MISSING')

ANTHROPIC_API_KEY 로드: OK


In [2]:
# 패키지 설치 (최초 1회)
!pip install langchain-ollama langchain langchain-core langgraph python-dotenv


  Using cached langchain_anthropic-1.4.0-py3-none-any.whl.metadata (3.2 kB)
  Attempting uninstall: langchain-anthropic
    Found existing installation: langchain-anthropic 0.3.22
    Uninstalling langchain-anthropic-0.3.22:
      Successfully uninstalled langchain-anthropic-0.3.22


---
## Part 1. LangGraph 핵심 구조 직접 이해

LangGraph는 에이전트를 **유향 그래프**로 표현합니다.
- 노드 = 처리 단계 (LLM 호출, 툴 실행, 사람 확인 등)
- 엣지 = 다음 노드 결정 (고정 또는 조건부)
- 상태 = 그래프를 흐르는 데이터

In [3]:
# ── LangGraph 핵심 임포트
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator

# ── State 정의: 그래프 전체를 흐르는 공유 메모리
class AgentState(TypedDict):
    messages: Annotated[List[dict], operator.add]  # 메시지 누적 (add reducer)
    task: str                                       # 현재 태스크
    result: str                                     # 최종 결과
    step_count: int                                 # 반복 횟수 (무한 루프 방지)

print('State 정의 완료')
print('AgentState fields:', list(AgentState.__annotations__.keys()))

State 정의 완료
AgentState fields: ['messages', 'task', 'result', 'step_count']


In [ ]:
# ── Ollama LLM 설정 (로컬 — 완전 무료)
# Ollama가 실행 중이어야 합니다 (ollama serve)
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

llm = ChatOllama(
    model="llama3.2",
    temperature=0,
)

def call_llm(messages: list, system: str = "") -> str:
    """Ollama 호출 헬퍼 (간단한 텍스트 응답용)"""
    lc_messages = []
    if system:
        lc_messages.append(SystemMessage(content=system))
    for m in messages:
        if isinstance(m, (HumanMessage, AIMessage, ToolMessage, SystemMessage)):
            lc_messages.append(m)
        elif isinstance(m, dict):
            content = m["content"] if isinstance(m["content"], str) else str(m["content"])
            if m["role"] == "user":
                lc_messages.append(HumanMessage(content=content))
            else:
                lc_messages.append(AIMessage(content=content))
    return llm.invoke(lc_messages).content

# 연결 테스트
test_response = call_llm([{"role": "user", "content": "안녕! 한 문장으로 답해줘."}])
print("Ollama 응답:", test_response)


---
## Part 2. 단일 에이전트 — ReAct 패턴

**ReAct** (Reasoning + Acting): LLM이 생각(Thought) → 행동(Action) → 관찰(Observation)을 반복

```
Thought: 이미지를 분석해야 한다
Action: vision_tool(image)
Observation: 이미지에 차량 3대, 사람 1명 검출됨
Thought: 결과를 요약해야 한다
Action: final_answer(summary)
```

In [ ]:
# ── 간단한 툴 정의 (나중에 실제 CV 모델로 교체)
from langchain_core.tools import tool

@tool
def analyze_image(image: str) -> str:
    """이미지를 분석하여 객체, 장면, 주요 특징을 반환합니다."""
    # 나중에 실제 YOLO/SAM으로 교체할 placeholder
    return f'[Vision Tool] 분석 완료: {image}에서 차량 2대, 보행자 1명, 신호등 1개 감지됨'

@tool  
def search_knowledge_base(query: str) -> str:
    """지식 베이스에서 관련 정보를 검색합니다."""
    # 나중에 실제 ChromaDB RAG로 교체할 placeholder
    return f'[RAG Tool] 검색 결과: {query}에 관련된 도로교통법 기준 및 안전 지침 3건 발견'

@tool
def generate_report(analysis: str, context: str) -> str:
    """분석 결과와 컨텍스트를 기반으로 구조화된 리포트를 생성합니다."""
    return f'[Report Tool] 리포트 생성 완료\n분석: {analysis}\n컨텍스트: {context}'

TOOLS = [analyze_image, search_knowledge_base, generate_report]
TOOL_MAP = {t.name: t for t in TOOLS}

print('등록된 툴:', [t.name for t in TOOLS])

In [ ]:
# ── LLM에 툴 바인딩 (LangChain @tool → Ollama 자동 호환)
import json

llm_with_tools = llm.bind_tools(TOOLS)

print("툴 바인딩 완료:", [t.name for t in TOOLS])
print("LLM 모델:", llm.model)


In [ ]:
# ── ReAct 에이전트 노드 구현

SYSTEM_PROMPT = """당신은 멀티모달 분석 에이전트입니다.
사용자가 이미지/장면에 대한 질문을 하면, 제공된 툴을 사용하여 단계적으로 분석하고 리포트를 작성하세요.

작업 순서:
1. analyze_image: 이미지/장면 분석
2. search_knowledge_base: 관련 지식 검색
3. generate_report: 최종 리포트 생성
"""

def _to_lc_messages(messages: list, system: str = "") -> list:
    """dict/LangChain 메시지 혼합 → LangChain 메시지 리스트 변환"""
    result = []
    if system:
        result.append(SystemMessage(content=system))
    for m in messages:
        if isinstance(m, (HumanMessage, AIMessage, ToolMessage, SystemMessage)):
            result.append(m)
        elif isinstance(m, dict):
            role = m["role"]
            content = m["content"]
            if isinstance(content, list):
                for item in content:
                    result.append(ToolMessage(
                        content=item.get("content", ""),
                        tool_call_id=item.get("tool_use_id", item.get("tool_call_id", ""))
                    ))
            elif role == "user":
                result.append(HumanMessage(content=str(content)))
            else:
                result.append(AIMessage(content=str(content)))
    return result


def agent_node(state: AgentState) -> AgentState:
    """LLM이 툴 호출 여부를 결정하는 메인 노드"""
    step = state.get("step_count", 0)
    lc_messages = _to_lc_messages(state["messages"], system=SYSTEM_PROMPT)
    response = llm_with_tools.invoke(lc_messages)

    tc_names = [tc["name"] for tc in response.tool_calls] if response.tool_calls else []
    print(f"  [Agent step {step+1}] tool_calls: {tc_names if tc_names else '없음'}")

    return {"messages": [response], "step_count": step + 1}


def tool_node(state: AgentState) -> AgentState:
    """LLM이 요청한 툴을 실행하는 노드"""
    last_message = state["messages"][-1]
    tool_results = []

    for tool_call in last_message.tool_calls:
        tool_name    = tool_call["name"]
        tool_input   = tool_call["args"]
        tool_call_id = tool_call["id"]

        if tool_name in TOOL_MAP:
            result = TOOL_MAP[tool_name].invoke(tool_input)
            print(f"  -> 툴 실행: {tool_name}")
            print(f"  <- 결과: {str(result)[:100]}")
        else:
            result = f"툴 {tool_name}을 찾을 수 없습니다"

        tool_results.append(ToolMessage(content=str(result), tool_call_id=tool_call_id))

    return {"messages": tool_results}

print("노드 함수 정의 완료")


In [ ]:
# ── 조건부 엣지: 툴 호출 여부 판단

def should_continue(state: AgentState) -> str:
    if state.get("step_count", 0) >= 10:
        print("최대 스텝 도달 -> 종료")
        return END

    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", None)
    if tool_calls:
        return "tool_node"
    return END

print("조건부 엣지 정의 완료")


In [ ]:
# ── StateGraph 구성

workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node('agent', agent_node)
workflow.add_node('tool_node', tool_node)

# 엣지 연결
workflow.set_entry_point('agent')                     # 시작점
workflow.add_conditional_edges(
    'agent',
    should_continue,                                  # 조건 함수
    {'tool_node': 'tool_node', END: END}              # 분기 매핑
)
workflow.add_edge('tool_node', 'agent')               # 툴 실행 후 agent로 복귀

# 그래프 컴파일
app = workflow.compile()

print('그래프 컴파일 완료')
print('노드:', ['agent', 'tool_node'])
print('흐름: START → agent → (tool_node → agent)* → END')

In [ ]:
# ── 그래프 시각화
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print('시각화 실패 (mermaid 미설치):', e)
    print('그래프 구조 (텍스트):')
    print(app.get_graph().draw_mermaid())

In [ ]:
# ── 단일 에이전트 실행 테스트

initial_state = {
    "messages": [HumanMessage(content="교차로 이미지를 분석하고 안전 위험 요소를 리포트로 작성해줘")],
    "task": "교차로 안전 분석",
    "result": "",
    "step_count": 0
}

print("=== 에이전트 실행 시작 ===")
print(f"입력: {initial_state['messages'][0].content}")
print()

final_state = app.invoke(initial_state)

print()
print("=== 최종 응답 ===")
last_msg = final_state["messages"][-1]
print(last_msg.content if hasattr(last_msg, "content") else str(last_msg))
print(f"\n총 스텝: {final_state['step_count']}")


---
## Part 3. Human-in-the-Loop 패턴

에이전트가 중요한 결정을 내리기 전에 **사람의 확인**을 받는 패턴.
LangGraph의 `interrupt_before`로 특정 노드 전에 실행을 멈출 수 있습니다.

```
agent → [사람 확인] → tool_node → agent → END
         ↑ interrupt
```

In [ ]:
# ── Human-in-the-Loop: Checkpointer + interrupt_before
from langgraph.checkpoint.memory import MemorySaver

# Checkpointer: 상태 스냅샷 저장 (thread_id로 대화 세션 관리)
checkpointer = MemorySaver()

# interrupt_before='tool_node': 툴 실행 전 사람 확인 요청
app_hitl = workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=['tool_node']
)

print('Human-in-the-Loop 그래프 컴파일 완료')
print('interrupt_before: tool_node (툴 실행 전 멈춤)')

In [ ]:
# -- HITL 실행 흐름
config = {"configurable": {"thread_id": "session_001"}}

print("=== 1단계: 에이전트 실행 (툴 호출 전 멈춤) ===")
state_snapshot = app_hitl.invoke(initial_state, config)

current_state = app_hitl.get_state(config)
print(f"다음 실행 노드: {current_state.next}")
print()

last_msg = current_state.values["messages"][-1]
if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        print(f"에이전트가 호출하려는 툴: {tc['name']}")
        print(f"입력값: {tc['args']}")

print()
# Jupyter 없이도 실행 가능하도록 자동 승인
# 실제 Jupyter 환경에서는: user_approval = input("허용할까요? (y/n): ")
user_approval = "y"
print(f"[자동 승인] user_approval = '{user_approval}'")


In [ ]:
# 2단계: 승인 시 재개, 거부 시 종료
if user_approval.lower() == "y":
    print("=== 2단계: 승인 → 실행 재개 ===")
    final_state = app_hitl.invoke(None, config)  # None: 현재 상태에서 재개
    
    last_msg = final_state["messages"][-1]
    # AIMessage 객체는 .content 속성으로 접근
    content = last_msg.content if hasattr(last_msg, "content") else str(last_msg)
    if isinstance(content, list):
        for block in content:
            if hasattr(block, "type") and block.type == "text":
                print("최종 결과:", block.text)
            elif isinstance(block, dict) and block.get("type") == "text":
                print("최종 결과:", block.get("text", ""))
    else:
        print("최종 결과:", content)
else:
    print("사용자가 툴 실행을 거부했습니다. 에이전트 중단.")

---
## Part 4. 멀티 에이전트 — Supervisor 패턴

여러 전문화된 에이전트를 **Supervisor**가 조율하는 구조.

```
              Supervisor
             ↙    ↓    ↘
       Vision   RAG   Report
       Agent   Agent  Writer
```

Supervisor가 태스크를 보고 어떤 에이전트에게 위임할지 결정합니다.

In [ ]:
# ── 멀티 에이전트 State 정의
from typing import Literal

AGENTS = ['vision_analyst', 'rag_retriever', 'report_writer', 'FINISH']

class MultiAgentState(TypedDict):
    messages: Annotated[List[dict], operator.add]
    next_agent: str          # Supervisor가 다음에 호출할 에이전트
    vision_result: str       # Vision Analyst 결과
    rag_result: str          # RAG Retriever 결과
    final_report: str        # Report Writer 결과

print('멀티 에이전트 State 정의 완료')
print('에이전트 목록:', AGENTS)

In [ ]:
# ── Supervisor 노드

SUPERVISOR_PROMPT = """당신은 멀티모달 분석 파이프라인을 조율하는 Supervisor입니다.

팀 구성:
- vision_analyst: 이미지/영상에서 객체, 장면, 이상 상황을 탐지
- rag_retriever: 지식 베이스에서 관련 기준, 규정, 레퍼런스를 검색
- report_writer: 분석 결과를 구조화된 리포트로 작성
- FINISH: 모든 작업이 완료되어 결과를 반환

현재 상황을 보고 다음에 호출할 에이전트 하나를 선택하세요.
반드시 다음 중 하나로만 답하세요: vision_analyst, rag_retriever, report_writer, FINISH
"""

def supervisor_node(state: MultiAgentState) -> MultiAgentState:
    first_msg = state["messages"][0]
    task_content = first_msg.content if hasattr(first_msg, "content") else str(first_msg)
    status = f"""
현재 상태:
- Vision 분석: {"완료" if state.get("vision_result") else "미완료"}
- RAG 검색: {"완료" if state.get("rag_result") else "미완료"}
- 리포트 작성: {"완료" if state.get("final_report") else "미완료"}

원본 태스크: {task_content}
"""
    response = call_llm(
        [{"role": "user", "content": status + "\n다음에 호출할 에이전트는?"}],
        system=SUPERVISOR_PROMPT
    )
    next_agent = "FINISH"
    for agent in AGENTS:
        if agent.lower() in response.lower():
            next_agent = agent
            break
    print(f"  [Supervisor] -> {next_agent}")
    return {"next_agent": next_agent}

print("Supervisor 노드 정의 완료")


In [ ]:
# ── 전문 에이전트 노드들

def vision_analyst_node(state: MultiAgentState) -> MultiAgentState:
    first_msg = state["messages"][0]
    task = first_msg.content if hasattr(first_msg, "content") else str(first_msg)
    response = call_llm(
        [{"role": "user", "content": f"다음 장면을 분석해주세요: {task}"}],
        system="당신은 컴퓨터 비전 전문가입니다. 이미지에서 객체, 위험 요소, 장면 특성을 상세히 분석합니다."
    )
    print(f"  [Vision Analyst] 완료 ({len(response)}자)")
    return {
        "vision_result": response,
        "messages": [AIMessage(content=f"[Vision] {response}")]
    }

def rag_retriever_node(state: MultiAgentState) -> MultiAgentState:
    vision = state.get("vision_result", "")
    response = call_llm(
        [{"role": "user", "content": f"다음 분석 결과와 관련된 안전 기준과 규정을 알려주세요: {vision[:200]}"}],
        system="당신은 도로교통 안전 전문가입니다. 법규, 기준, 권장사항을 제공합니다."
    )
    print(f"  [RAG Retriever] 완료 ({len(response)}자)")
    return {
        "rag_result": response,
        "messages": [AIMessage(content=f"[RAG] {response}")]
    }

def report_writer_node(state: MultiAgentState) -> MultiAgentState:
    vision = state.get("vision_result", "없음")
    rag   = state.get("rag_result", "없음")
    prompt = f"""다음 정보를 바탕으로 구조화된 분석 리포트를 작성해주세요.

## 비전 분석 결과
{vision}

## 관련 규정/기준
{rag}

리포트 형식: 요약 / 발견사항 / 위험도 평가 / 권고사항"""
    response = call_llm(
        [{"role": "user", "content": prompt}],
        system="당신은 기술 리포트 작성 전문가입니다. 명확하고 구조화된 분석 보고서를 작성합니다."
    )
    print(f"  [Report Writer] 완료 ({len(response)}자)")
    return {
        "final_report": response,
        "messages": [AIMessage(content=f"[Report] {response}")]
    }

print("전문 에이전트 노드 정의 완료")


In [ ]:
# ── 멀티 에이전트 그래프 구성

multi_workflow = StateGraph(MultiAgentState)

# 노드 추가
multi_workflow.add_node('supervisor', supervisor_node)
multi_workflow.add_node('vision_analyst', vision_analyst_node)
multi_workflow.add_node('rag_retriever', rag_retriever_node)
multi_workflow.add_node('report_writer', report_writer_node)

# 시작점: Supervisor
multi_workflow.set_entry_point('supervisor')

# Supervisor의 결정에 따라 분기
multi_workflow.add_conditional_edges(
    'supervisor',
    lambda state: state['next_agent'],
    {
        'vision_analyst': 'vision_analyst',
        'rag_retriever': 'rag_retriever',
        'report_writer': 'report_writer',
        'FINISH': END
    }
)

# 각 에이전트 완료 후 Supervisor로 복귀
for agent in ['vision_analyst', 'rag_retriever', 'report_writer']:
    multi_workflow.add_edge(agent, 'supervisor')

# 컴파일
multi_app = multi_workflow.compile()

print('멀티 에이전트 그래프 컴파일 완료')
print('흐름: START → supervisor → (vision/rag/report)* → FINISH')

In [ ]:
# ── 멀티 에이전트 실행

multi_initial = {
    "messages": [HumanMessage(content="비가 오는 밤 교차로에서 보행자와 차량이 교차하는 장면을 분석하고 안전 위험 요소 리포트를 작성해줘")],
    "next_agent": "",
    "vision_result": "",
    "rag_result": "",
    "final_report": ""
}

print("=== 멀티 에이전트 실행 시작 ===")
print(f"태스크: {multi_initial['messages'][0].content}")
print()

final_multi_state = multi_app.invoke(multi_initial)

print()
print("=" * 60)
print("최종 리포트")
print("=" * 60)
print(final_multi_state.get("final_report", "리포트 없음"))


---
## 정리

| 개념 | 설명 | 구현 위치 |
|------|------|----------|
| StateGraph | 에이전트 상태머신의 틀 | `workflow = StateGraph(AgentState)` |
| Node | 상태를 받아 처리하는 함수 | `agent_node`, `tool_node` |
| Conditional Edge | 조건부 분기 | `should_continue` |
| Checkpointer | 상태 저장/복원 | `MemorySaver` |
| interrupt_before | HITL 멈춤 포인트 | `compile(interrupt_before=[...])` |
| Supervisor 패턴 | 멀티 에이전트 조율 | `supervisor_node` |

## 다음 노트북
**02_vision_tools.ipynb**: 실제 YOLO/SAM/DepthAnything을 LangGraph Tool로 래핑